In [ ]:
# Core libraries
import numpy as np
import matplotlib.pyplot as plt

# Qiskit
from qiskit import QuantumCircuit, Aer, transpile
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

# Noise model tools
from qiskit.providers.fake_provider import FakeAthens
from qiskit_aer.noise import NoiseModel

# Linear algebra reference
from numpy.linalg import solve, eig

# The HHL Algorithm – Theoretical Background

The HHL algorithm solves: $A\vec{x} = \vec{b}$

by preparing quantum states: $|b\rangle \longrightarrow |x\rangle \propto A^{-1}|b\rangle$

1.1 Core Steps of HHL

1. State Preparation: $|b\rangle = \sum_i \beta_i |u_i\rangle$

2. Quantum Phase Estimation (QPE): $|u_i\rangle \longrightarrow |\lambda_i\rangle |u_i\rangle$

3. Controlled Rotation

$$
|\lambda_i\rangle |0\rangle
\longrightarrow
|\lambda_i\rangle
\left(
\sqrt{1 - \frac{C^2}{\lambda_i^2}}\,|0\rangle
+
\frac{C}{\lambda_i}\,|1\rangle
\right)
$$

4. Uncompute QPE

Apply the inverse of the QPE circuit to disentangle the eigenvalue register.

5. Post-select ancilla = 1

Measure the ancilla qubit and retain outcomes where it is found in the state $|1\rangle$.

Final State

$$|x\rangle \propto \sum_i \frac{\beta_i}{\lambda_i} |u_i\rangle$$

# Problem 1 – 4×4 Matrix

In [ ]:
# Define Matrix and Analyze Spectrum
A = np.array([], dtype=complex)

eigvals, eigvecs = eig(A)

print("Eigenvalues:")
print(eigvals)

📌 Tasks:

- Verify if A is Hermitian.
- Check condition number.
- Determine if scaling is required for HHL.

In [ ]:
# Basis vector |b>
basis_vectors = [np.array([],dtype=complex),...]

In [ ]:
basis_vectors = []

Classical Reference Solution

⚠️ Do not use these values in your HHL circuit directly.

In [ ]:
def classical_solution(A, b):
    return solve(A, b)

for i, b in enumerate(basis_vectors):
    x = classical_solution(A, b)
    print(f"Solution for basis vector {i}:")
    print(x)

In [ ]:
def build_hhl_circuit(A, b, num_phase_qubits=2):
    
    n = int(np.log2(len(b)))
    
    qc = QuantumCircuit(
        1 + num_phase_qubits + n,  # ancilla + phase + system
        name="HHL"
    )
    
    # -------------------------
    # Step 1: State Preparation
    # -------------------------
    
    # TODO: Encode |b> into system register
    
    
    # -------------------------
    # Step 2: QPE
    # -------------------------
    
    # TODO:
    # 1. Apply Hadamards on phase register
    # 2. Apply controlled-e^{iAt}
    # 3. Apply inverse QFT
    
    
    # -------------------------
    # Step 3: Controlled Rotation
    # -------------------------
    
    # TODO:
    # Implement controlled R_y depending on eigenvalue register
    
    
    # -------------------------
    # Step 4: Uncompute QPE
    # -------------------------
    
    # TODO
    
    
    return qc

Ideal Simulator Execution

In [ ]:
sim = AerSimulator(method="statevector")

qc = build_hhl_circuit(A, basis_vectors[0])

tqc = transpile(qc, sim)
result = sim.run(tqc).result()

state = result.get_statevector()
print(state)

📌 Tasks:

- Extract |x⟩ from post-selection.
- Normalize reconstructed solution.
- Compare with classical result.

Noisy Simulation

In [ ]:
backend = FakeAthens()
noise_model = NoiseModel.from_backend(backend)

noisy_sim = AerSimulator(noise_model=noise_model)

tqc_noisy = transpile(qc, noisy_sim)
result_noisy = noisy_sim.run(tqc_noisy).result()

📌 Tasks:

- Compute fidelity between ideal and noisy outputs.
- Analyze degradation.

Gate Count Analysis

In [ ]:
tqc = transpile(qc, optimization_level=0)
print(tqc.count_ops())

tqc_trans = transpile(qc, optimization_level=3)
print(tqc_trans.count_ops())

📌 Tasks:

- Count single-qubit gates.
- Count two-qubit gates.
- Identify if Toffoli gates appear.
- If yes, how many?

## Second Matrix (Block Form)

Construct Matrix

In [ ]:
I7 = np.eye(7)

A2 = np.block([[],[],[]])

📌 Tasks:

- Determine dimension.
- Determine number of qubits required.
- Estimate circuit width before implementation.

# Problem 2 – Order Finding Unitary

From Nielsen & Chuang (10th Anniversary Edition):

Consider unitary arising from order finding for: $x=4,N=15$

Construct Modular Multiplication Operator

In [ ]:
def modular_multiplication_unitary(x, N):
    # TODO:
    # Build permutation matrix representing:
    # |y> -> |xy mod N>
    pass

HHL-style Circuit

In [ ]:
def build_order_hhl_circuit(U):
    qc = QuantumCircuit(...)
    
    # TODO:
    # Implement:
    # QPE
    # Controlled rotation
    # Inverse QPE
    
    return qc

📌 Tasks:

Derive rotation angle: $θ=2\sin^{−1}(C/\lambda)$

Write matrix:
$$R = \begin{pmatrix}
\cos(\theta/2) & 
\sin(\theta/2) \\\
−\sin(\theta/2) & 
\cos(\theta/2)
\end{pmatrix}$$
 Write explicitly in your submission.

## Margolus Gate Replacement

Margolus gate: 

$$G = \begin{pmatrix}
\cos(\pi/8) & 
\sin(\pi/8) \\\
−\sin(\pi/8) & 
\cos(\pi/8)
\end{pmatrix}$$

📌 Tasks:

- Replace Toffoli with Margolus.
- Analyze difference.
- Compute output |x⟩ manually.
- Write final column vector explicitly (not Python array).